## Final Project Submission

Please fill out:
* Student name: 
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: 
* Blog post URL:


# Movie Studio Analysis

## Business Understanding

The goal of this project is to analyze movie industry data and provide actionable recommendations to a company planning to create a new movie studio.

We will analyze movie characteristics, ratings, genres, runtime, and box office performance to identify factors associated with successful movies.

## Data Understanding

The analysis will use movie industry datasets containing information about movies, ratings, genres, runtime, and box office performance.

The main datasets we will work with are:

- IMDb movie data stored in `im.db`
- Box Office Mojo data stored in `bom.movie_gross.csv`
- Additional movie datasets available in the `zippedData` folder

We will first inspect the available data, understand the columns and data types, and then determine which datasets are most useful for answering our business questions.

In [62]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

### Load the Box Office Mojo Dataset

We load the Box Office Mojo dataset into a pandas DataFrame so that we can inspect its structure and assess its data quality before cleaning.

In [66]:
# Load the Box Office Mojo movie gross dataset
bom = pd.read_csv("zippedData/bom.movie_gross.csv")

### Preview the Dataset

We display the first few records to understand the type of information contained in the dataset.

In [68]:
# Display the first five rows of the dataset
bom.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


### Dataset Dimensions

We check the number of rows and columns in the Box Office Mojo dataset to understand its size before carrying out further inspection and cleaning.

In [69]:
# Check the number of rows and columns
bom.shape

(3387, 5)

### Dataset Structure

We inspect the dataset structure to identify the column names, number of non-null observations, and data types. This helps us identify columns that may require cleaning.

In [70]:
# Display column names, non-null counts, and data types
bom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


### Missing Values

We check each column for missing values to identify incomplete records and determine which variables require attention during the cleaning stage.

In [71]:
# Count missing values in each column
bom.isna().sum()

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

### Duplicate Records

We check for completely duplicated rows to ensure that duplicate movie records do not cause movies or revenue to be counted more than once.

In [72]:
# Count completely duplicated rows
bom.duplicated().sum()

0

### Inspecting Box Office Revenue

The `domestic_gross` and `foreign_gross` columns contain box-office revenue. We inspect their values and data types before deciding how they should be cleaned and prepared for analysis.

In [73]:
# Inspect the data types of the revenue columns
bom[["domestic_gross", "foreign_gross"]].dtypes

domestic_gross    float64
foreign_gross      object
dtype: object

In [74]:
# Display sample revenue values
bom[["domestic_gross", "foreign_gross"]].head(10)

,domestic_gross,foreign_gross
0,415000000.0,652000000
1,334200000.0,691300000
2,296000000.0,664300000
3,292600000.0,535700000
4,238700000.0,513900000
5,300500000.0,398000000
6,312400000.0,311500000
7,200800000.0,391000000
8,251500000.0,291600000
9,217600000.0,277300000


## BOM Dataset Inspection Findings

The Box Office Mojo dataset contains **3,387 movie records and 5 variables**: `title`, `studio`, `domestic_gross`, `foreign_gross`, and `year`.

The inspection shows that `title` and `year` have complete records, while `studio`, `domestic_gross`, and `foreign_gross` contain missing values.

- `studio` has **5 missing values**.
- `domestic_gross` has **28 missing values**.
- `foreign_gross` has **1,350 missing values**.

The data types also reveal an important issue. `domestic_gross` is stored as a numeric (`float64`) variable, while `foreign_gross` is stored as an `object`. Therefore, `foreign_gross` will require further cleaning and conversion before it can be reliably used in numerical calculations.

The duplicate check returned **0 completely duplicated rows**, indicating that no identical records need to be removed from the dataset.

Overall, the inspection identifies missing values and an inconsistent data type in `foreign_gross` as the main data-quality issues to address during the cleaning stage.

## Data Preparation

Based on the inspection findings, the BOM dataset requires cleaning before it can be used for analysis.

The cleaning process will focus on:

1. Handling missing values in `studio`, `domestic_gross`, and `foreign_gross`.
2. Converting `foreign_gross` from an object to a numeric variable.
3. Checking the cleaned values for consistency.
4. Preparing the revenue variables for calculating total box-office performance.
5. Preserving valid movie records while avoiding unnecessary data loss.

## Data Cleaning

### Handling Missing Studio Values

The `studio` column identifies the movie's production or distribution studio.

Our inspection identified 5 records with missing studio values. Before deciding how to handle these records, we inspect them to determine whether the studio information can be recovered from the available data or whether the records should be excluded from studio-level analysis.

In [75]:
# Display movies with missing studio values
bom[bom["studio"].isna()]

,title,studio,domestic_gross,foreign_gross,year
210,Outside the Law (Hors-la-loi),NaN,96900.0,3300000,2010
555,Fireflies in the Garden,NaN,70600.0,3300000,2011
933,Keith Lemon: The Film,NaN,NaN,4000000,2012
1862,Plot for Peace,NaN,7100.0,NaN,2014
2825,Secret Superstar,NaN,NaN,122000000,2017


In [76]:
# Display the title, gross revenue, and year for movies with missing studios
bom.loc[
    bom["studio"].isna(),
    ["title", "domestic_gross", "foreign_gross", "year"]
]

,title,domestic_gross,foreign_gross,year
210,Outside the Law (Hors-la-loi),96900.0,3300000,2010
555,Fireflies in the Garden,70600.0,3300000,2011
933,Keith Lemon: The Film,NaN,4000000,2012
1862,Plot for Peace,7100.0,NaN,2014
2825,Secret Superstar,NaN,122000000,2017


### Handling Missing Studio Values

The inspection identified 5 movie records with missing values in the `studio` column.

The affected records contain valid movie information and should not be removed simply because the studio is unavailable. Since the available BOM data does not provide enough information to reliably determine the missing studios, we will replace the missing studio values with `"Unknown"`.

This approach preserves the movie records while clearly identifying that the studio information was unavailable. The `"Unknown"` category can be excluded from studio-level comparisons where appropriate.

In [77]:
# Replace missing studio values with "Unknown"
bom["studio"] = bom["studio"].fillna("Unknown")

### Verify Studio Cleaning

We verify that all missing values in the `studio` column have been handled successfully.

In [78]:
# Check the number of remaining missing studio values
bom["studio"].isna().sum()

0

In [81]:
# Confirm that five records were assigned to the "Unknown" category
bom["studio"].value_counts().get("Unknown", 0)

5

In [80]:
# Confirm that the number of rows remains unchanged after cleaning
bom.shape

(3387, 5)

### Studio Cleaning Result

The 5 missing values in the `studio` column were replaced with `"Unknown"`.

No movie records were removed during this process, so the dataset remains at 3,387 rows. Using an `"Unknown"` category preserves valid movie records while clearly indicating that the studio information was unavailable.

These `"Unknown"` records will be considered appropriately when performing studio-level analysis.

### Handling Missing Domestic Gross Values

The inspection identified 28 missing values in the `domestic_gross` column.

Domestic gross is an important measure of box-office performance, so we should not replace missing values with an arbitrary value such as zero without first investigating the affected records.

We will inspect the affected movies and determine an appropriate treatment while avoiding unnecessary loss of valid records.

In [82]:
# Display movies with missing domestic gross values
bom[bom["domestic_gross"].isna()]

,title,studio,domestic_gross,foreign_gross,year
230,It's a Wonderful Afterlife,UTV,NaN,1300000,2010
298,Celine: Through the Eyes of the World,Sony,NaN,119000,2010
302,White Lion,Scre.,NaN,99600,2010
306,Badmaash Company,Yash,NaN,64400,2010
327,Aashayein (Wishes),Relbig.,NaN,3800,2010
537,Force,FoxS,NaN,4800000,2011
713,Empire of Silver,NeoC,NaN,19000,2011
871,Solomon Kane,RTWC,NaN,19600000,2012
928,The Tall Man,Imag.,NaN,5200000,2012
933,Keith Lemon: The Film,Unknown,NaN,4000000,2012


In [83]:
# Display relevant information for movies with missing domestic gross
bom.loc[
    bom["domestic_gross"].isna(),
    ["title", "studio", "foreign_gross", "year"]
]


,title,studio,foreign_gross,year
230,It's a Wonderful Afterlife,UTV,1300000,2010
298,Celine: Through the Eyes of the World,Sony,119000,2010
302,White Lion,Scre.,99600,2010
306,Badmaash Company,Yash,64400,2010
327,Aashayein (Wishes),Relbig.,3800,2010
537,Force,FoxS,4800000,2011
713,Empire of Silver,NeoC,19000,2011
871,Solomon Kane,RTWC,19600000,2012
928,The Tall Man,Imag.,5200000,2012
933,Keith Lemon: The Film,Unknown,4000000,2012


In [84]:
# Count the remaining missing domestic gross values
bom["domestic_gross"].isna().sum()

28

### Handling Missing Domestic Gross Values

The inspection identified 28 missing values in the `domestic_gross` column.

The affected records are valid movie observations and several have reported `foreign_gross` values. Therefore, the missing domestic gross values should not automatically be interpreted as zero.

Because the actual domestic gross cannot be reliably determined from the BOM dataset, we will retain these records and leave the missing values as `NaN`. This avoids introducing inaccurate revenue values or unnecessarily removing valid movie records.

For analyses that specifically require domestic gross, pandas will exclude these missing observations from calculations by default.

In [85]:
# Confirm that the 28 domestic gross values are still missing
bom["domestic_gross"].isna().sum()

28

In [86]:
# Confirm that no movie records were removed
bom.shape

(3387, 5)

### Domestic Gross Cleaning Result

The 28 missing values in `domestic_gross` were retained as missing values because the actual domestic revenue cannot be reliably determined from the available data.

No rows were removed and no artificial revenue values were introduced. The dataset therefore remains at 3,387 records.

For analyses that require domestic gross, calculations will be based only on records with available domestic gross values.

### Handling Foreign Gross Values

The `foreign_gross` column contains foreign box-office revenue. During inspection, we identified 1,350 missing values and found that the column is stored as an `object` rather than a numeric data type.

Before performing numerical analysis, we need to convert the available foreign gross values to a numeric format. Missing values will remain as missing because there is not enough information in this dataset to reliably determine the actual foreign revenue.

In [87]:
# Display sample values from the foreign gross column
bom["foreign_gross"].head(20)

0     652000000
1     691300000
2     664300000
3     535700000
4     513900000
5     398000000
6     311500000
7     391000000
8     291600000
9     277300000
10    330000000
11    311300000
12    275400000
13    228000000
14    182500000
15    245600000
16    222400000
17    173500000
18    216400000
19    187900000
Name: foreign_gross, dtype: object

In [88]:
# Check the current data type
bom["foreign_gross"].dtype

dtype('O')

In [89]:
# Convert foreign gross values to numeric
# Invalid or missing values are converted to NaN
bom["foreign_gross"] = pd.to_numeric(
    bom["foreign_gross"],
    errors="coerce"
)

### Verify Foreign Gross Conversion

After conversion, we check the data type and missing-value count to confirm that the revenue variable is ready for numerical analysis.

In [90]:
# Check the new data type
bom["foreign_gross"].dtype

dtype('float64')

In [91]:
# Check the number of missing foreign gross values
bom["foreign_gross"].isna().sum()

1355

In [92]:
# Display summary statistics for foreign gross
bom["foreign_gross"].describe()

count    2.032000e+03
mean     7.505704e+07
std      1.375294e+08
min      6.000000e+02
25%      3.775000e+06
50%      1.890000e+07
75%      7.505000e+07
max      9.605000e+08
Name: foreign_gross, dtype: float64

### Identifying Non-Numeric Foreign Gross Values

After converting `foreign_gross` to a numeric data type, we investigate the values that could not be converted. This helps us understand why the number of missing values increased after conversion and ensures that the cleaning process is properly documented.

In [93]:
# Reload the original foreign gross values for comparison
bom_original = pd.read_csv("zippedData/bom.movie_gross.csv")

# Identify values that were present but could not be converted to numeric
original_foreign = bom_original["foreign_gross"]

non_numeric = original_foreign[
    original_foreign.notna() &
    pd.to_numeric(original_foreign, errors="coerce").isna()
]

non_numeric

1872    1,131.6
1873    1,019.4
1874    1,163.0
2760    1,010.0
3079    1,369.5
Name: foreign_gross, dtype: object

### Cleaning Non-Numeric Foreign Gross Values

The investigation identified 5 `foreign_gross` values that could not initially be converted to numeric values.

These values contain commas as thousands separators, for example `1,131.6`. The values represent valid numerical amounts, so they should not be treated as missing.

We will remove the commas and convert these values to numeric format.

In [98]:
# Restore the original foreign gross values
bom["foreign_gross"] = bom_original["foreign_gross"]

# Remove thousands separators and convert the column to numeric
bom["foreign_gross"] = (
    bom["foreign_gross"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

In [99]:
# Confirm that foreign gross is now numeric
bom["foreign_gross"].dtype

dtype('float64')

In [100]:
# Check the number of missing foreign gross values after cleaning
bom["foreign_gross"].isna().sum()

1350

In [101]:
# Confirm that the previously non-numeric values were successfully converted
bom.loc[
    [1872, 1873, 1874, 2760, 3079],
    "foreign_gross"
]

1872    1131.6
1873    1019.4
1874    1163.0
2760    1010.0
3079    1369.5
Name: foreign_gross, dtype: float64

### Foreign Gross Cleaning Result

The `foreign_gross` column was successfully converted from an object data type to a numeric (`float64`) data type.

During the cleaning process, 5 values containing commas as thousands separators were identified. These values were cleaned and successfully converted to numeric values rather than being treated as missing.

After cleaning, the dataset contains 1,350 missing `foreign_gross` values, which matches the original missing-value count.

The remaining missing values were retained as `NaN` because their actual foreign box-office revenue cannot be reliably determined from the available dataset.